# EasyOCR (Inggris + Indonesia)
        **vision › ocr** · notebook `easyocr_en_id`

        **Kapan dipakai:** Membaca teks dari gambar tanpa melatih; model deteksi + pengenalan siap pakai, offline setelah bobot diunduh.

        **Jebakan umum:** Teks miring/kecil/berlatar ramai menurunkan akurasi; praproses (grayscale, perbesar) membantu. Evaluasi di sini memakai gambar teks SINTETIS buatan sendiri.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

> Evaluasi memakai **gambar teks sintetis** (dirender sendiri, ditandai). Bobot dibaca dari `pretrained/easyocr/` (offline).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
MODEL_KEY = "easyocr_en_id"; OUT = out_dir("vision", "ocr", MODEL_KEY); SEED = 42; set_seed(SEED)
WORDS = ["pasar", "harga", "beras", "kopi", "Jakarta", "market", "price", "coffee", "diskon", "gratis", "invoice", "Surabaya", "data mining", "kompetisi", "model", "hasil"]
NOISE, N = 18, 40

## 2. Buat gambar teks sintetis (kata + noise + rotasi kecil)
Karena kita tahu teks aslinya, **CER** (Character Error Rate) bisa dihitung.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
rng = np.random.default_rng(SEED)
font = ImageFont.load_default(size=36)
def render(word):
    im = Image.new("L", (360, 90), 255); ImageDraw.Draw(im).text((20, 20), word, fill=0, font=font)
    im = im.rotate(float(rng.uniform(-3, 3)), expand=False, fillcolor=255)
    a = np.asarray(im, float) + rng.normal(0, NOISE, im.size[::-1]); return np.clip(a, 0, 255).astype(np.uint8)
truth = [str(rng.choice(WORDS)) for _ in range(N)]; imgs = [render(w) for w in truth]
fig, ax = plt.subplots(1, 4, figsize=(11, 1.6)); [(a.imshow(imgs[i], cmap="gray"), a.axis("off")) for i, a in enumerate(ax)]; plt.tight_layout(); plt.show()

## 3. Jalankan EasyOCR (model lokal, tanpa unduhan)

In [ ]:
import easyocr, torch, time
from superdb.paths import ROOT as R
MD = R / "pretrained" / "easyocr"; assert MD.exists(), "bobot belum ada: python tools/download_pretrained_vision.py easyocr"
reader = easyocr.Reader(["en", "id"], gpu=torch.cuda.is_available(), model_storage_directory=str(MD), user_network_directory=str(MD), download_enabled=False, verbose=False)
t0 = time.time(); pred = [" ".join(reader.readtext(im, detail=0)) for im in imgs]; sec = time.time() - t0
print(f"{sec:.1f} dtk untuk {N} gambar | contoh:", list(zip(truth[:4], pred[:4])))

## 4. Evaluasi: CER dan akurasi kata persis
CER = jarak edit (Levenshtein) / panjang teks asli. 0 = sempurna.

In [ ]:
def lev(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1): prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[-1]
norm = lambda s: s.lower().strip()
cer = float(np.sum([lev(norm(t), norm(p)) for t, p in zip(truth, pred)]) / np.sum([len(t) for t in truth]))
exact = float(np.mean([norm(t) == norm(p) for t, p in zip(truth, pred)]))
metrics = {"CER": cer, "akurasi_kata_persis": exact, "detik_total": sec}; display(show_table(metrics))
save_json({"metrics": metrics, "contoh": list(zip(truth[:10], pred[:10]))}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Praproses meningkatkan OCR: grayscale, perbesar 2x, binarisasi adaptif, luruskan rotasi.
- Dokumen terstruktur (struk, KTP): sesudah OCR, ekstraksi kolom dengan regex/aturan.
- Evaluasi lomba biasanya CER/WER: fungsi `lev` di atas dapat dipakai langsung.